# Aula02 — LangChain

Uma seção por vídeo, cada uma autossuficiente. Requer o arquivo `.env` (veja `.example.env`).
Os `main.py` originais continuam em `video0X/`.

## video01

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv
import os

# CONFIGURAÇÃO DE AMBIENTE
# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

# 1. Configuração das credenciais (Mantenha suas chaves seguras)
# Validação para garantir que a chave de API foi carregada
if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
    raise ValueError("A variável de ambiente para a API_KEY não foi encontrada. Crie um arquivo .env e adicione-a.")

model = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)
parser = StrOutputParser()

# 2. Criação dos Componentes da Chain
prompt_gerador = ChatPromptTemplate.from_template(
    "Gere uma breve descrição técnica sobre o conceito de: {topico}"
)

# 3. Função customizada usando RunnableLambda (Sanitização)
prompt_revisor = ChatPromptTemplate.from_template(
    "Melhore o seguinte texto técnico, corrigindo eventuais termos e tornando-o mais formal:\n\n{texto_gerado}"
)

def limpar_texto(texto: str) -> str:
    # Simula uma limpeza ou remoção de caracteres indesejados
    return texto.strip().replace("  ", " ")

runnable_limpeza = RunnableLambda(limpar_texto)

# 4. Construindo a Chain com o Operador Pipe (|)
chain = (
    prompt_gerador
    | model 
    | parser 
    | runnable_limpeza
    | {"texto_gerado": RunnablePassthrough()}  # Mantém o dado vivo no dicionário para o próximo prompt
    | prompt_revisor 
    | model 
    | parser
)

print("=== 1. DEMONSTRANDO .invoke() ===")
resultado_invoke = chain.invoke({"topico": "LangChain Expression Language (LCEL)"})
print(resultado_invoke)
print("\n" + "="*40 + "\n")

print("=== 2. DEMONSTRANDO .stream() ===")
# Excelente para mostrar o surgimento de tokens em tempo real na gravação
for chunk in chain.stream({"topico": "RunnableParallel"}):
    print(chunk, end="", flush=True)
print("\n\n" + "="*40 + "\n")

print("=== 3. DEMONSTRANDO .batch() ===")
# Execução paralela nativa de múltiplos inputs
inputs = [
    {"topico": "Large Language Models"},
    {"topico": "API Idempotency"}
]

resultados_batch = chain.batch(inputs)
for i, res in enumerate(resultados_batch):
    print(f"Resultado {i+1}:\n{res}\n")

## video02

In [ ]:
import os
from pydantic import BaseModel, Field, field_validator
import unicodedata
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv

load_dotenv()

# Validação para garantir que a chave de API foi carregada
if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
    raise ValueError("A variável de ambiente para a API_KEY não foi encontrada. Crie um arquivo .env e adicione-a.")

# 1. Configuração do Modelo de Classificação Inteligente (Pydantic)
class RoteadorSuporte(BaseModel):
    """Classifica a intenção do usuário para direcionar ao setor correto.

    Normaliza o valor de `setor` para lowercase antes da validação, assim
    aceitando respostas como "Financeiro" ou "financeiro" vindas do LLM.
    """
    setor: str = Field(
        description="O setor para onde a dúvida deve ser enviada baseado no contexto."
    )

    @field_validator("setor", mode="before")
    def _normalize_setor(cls, v):
        if isinstance(v, str):
            # Remove acentos e normalize para facilitar mapeamento
            s = unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii")
            s = s.strip().lower()
            # Map common variations to canonical values
            if "finance" in s:
                return "financeiro"
            if "tecn" in s or "suporte" in s or "tecnico" in s:
                return "tecnico"
            return s
        return v

    @field_validator("setor")
    def _validate_setor(cls, v):
        if v not in ("financeiro", "tecnico"):
            raise ValueError("setor must be 'financeiro' or 'tecnico'")
        return v

model = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash", temperature=0)

llm_classificador = model.with_structured_output(RoteadorSuporte)

# 2. Definição das Sub-Chains Especializadas
prompt_financeiro = ChatPromptTemplate.from_template(
    "Você é um especialista financeiro. Responda a seguinte dúvida sobre faturamento de forma clara e cordial: {pergunta}"
)
prompt_tecnico = ChatPromptTemplate.from_template(
    "Você é um engenheiro de software sênior. Responda a seguinte dúvida de TI de forma técnica e objetiva: {pergunta}"
)

parser = StrOutputParser()

chain_financeira = prompt_financeiro | model | parser
chain_tecnica = prompt_tecnico | model | parser

# 3. Lógica do Roteador (Decisor Dinâmico)
def rotear_consulta(dados):
    intencao = dados["intencao"].setor
    pergunta_original = dados["pergunta"]
    
    print(f"\n[Roteador] -> Intenção detectada: {intencao.upper()}")
    
    if intencao == "financeiro":
        return chain_financeira.invoke({"pergunta": pergunta_original})
    elif intencao == "tecnico":
        return chain_tecnica.invoke({"pergunta": pergunta_original})

# 4. Orquestração da Chain de Roteamento Completa
# Usamos RunnableParallel de forma implícita passando o dicionário
chain_principal = {
    "intencao": ChatPromptTemplate.from_template("{pergunta}") | llm_classificador,
    "pergunta": RunnablePassthrough()
} | RunnableLambda(rotear_consulta)

# print("=== TESTE 1: INPUT FINANCEIRO ===")
# pergunta_1 = "Minha fatura veio duplicada este mês, como solicito o estorno?"
# resposta_1 = chain_principal.invoke({"pergunta": pergunta_1})
# print(f"Resposta Final:\n{resposta_1}")
# print("\n" + "="*40 + "\n")

print("=== TESTE 2: INPUT TÉCNICO ===")
pergunta_2 = "Estou recebendo erro 500 ao tentar conectar no banco PostgreSQL local."
resposta_2 = chain_principal.invoke({"pergunta": pergunta_2})
print(f"Resposta Final:\n{resposta_2}")

## video03

In [ ]:
import os
import logging
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
# [notebook] no LangChain 1.x o AgentExecutor foi movido para o pacote langchain-classic
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.globals import set_debug
from langchain_core.callbacks import BaseCallbackHandler

from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO)
set_debug(False)

# Checagem básica de credenciais (não interrompe, apenas alerta)
if not os.getenv("OPENAI_API_KEY"):
    logging.warning("OPENAI_API_KEY não encontrada. Chamadas ao OpenAI vão falhar se não configurada.")
if not os.getenv("GOOGLE_API_KEY"):
    logging.warning("GOOGLE_API_KEY não encontrada. Chamadas ao Google Generative AI vão falhar se não configurada.")


# 1. Definição de uma Custom Tool com Tipagem e Docstring Ricas
@tool
def calcular_faturamento_anual(empresa: str, receita_mensal: float) -> str:
    """Calcula a projeção de faturamento anual de uma empresa baseada na receita mensal."""
    # A docstring funciona como a descrição lida pelo agente para decidir o uso da ferramenta
    faturamento_projetado = receita_mensal * 12
    return f"A projeção de faturamento anual para a empresa {empresa} é de R$ {faturamento_projetado:,.2f}"

class LogFallbackHandler(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        nome_modelo = serialized.get("id", ["LLM"])[-1]
        print(f"\n🟡 [TENTATIVA] Chamando modelo: {nome_modelo}")

    def on_llm_error(self, error, **kwargs):
        print(f"🔴 [FALHA] Modelo falhou com o erro: {error}")
        print("🔄 [FALLBACK] Redirecionando para o modelo secundário...")

    def on_llm_end(self, response, **kwargs):
        print("🟢 [SUCESSO] LLM respondeu com sucesso!")

tools = [calcular_faturamento_anual]

# 2. Prompt Padrão
prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente financeiro corporativo de elite. Utilize as ferramentas fornecidas sempre que precisar de dados calculados."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# 3. Modelos
llm_primario = ChatOpenAI(model="gpt-5.4-mini", temperature=0, timeout=0.001, max_retries=0)
llm_secundario = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 4. Criação dos Agentes Individuais com Tool Calling Universal
agent_primario = create_tool_calling_agent(llm_primario, tools, prompt)
agent_secundario = create_tool_calling_agent(llm_secundario, tools, prompt)

# 5. Aplicação do Fallback no Nível do Agente (Runnable)
agent_resiliente = agent_primario.with_fallbacks([agent_secundario])

# 6. Runtime Executor
agent_executor = AgentExecutor(
    agent=agent_resiliente,
    tools=tools,
    verbose=True
)

print("=== EXECUTANDO O AGENTE AUTÔNOMO ===")
consulta = "A empresa AlfaTech fatura 25000 reais por mês. Qual a projeção anual deles?"

try:
    resposta = agent_executor.invoke({"input": consulta}, config={"callbacks": [LogFallbackHandler()]})
    print("\n=== RESPOSTA FINAL DO AGENTE ===")
    # Normaliza formatos de saída comuns
    if isinstance(resposta, dict):
        out = resposta.get("output") or resposta.get("text") or resposta.get("result") or str(resposta)
    else:
        out = str(resposta)
    print(out)
except Exception as e:
    logging.exception("Falha na invocação do agente")
    print("Erro ao executar o agente:", e)